In [55]:
# !nvidia-smi   # confirm it now says T4, not P100
import torch
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

import os

REPO_DIR = "/kaggle/working/MobileNetV2-Compression"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/lazylettuce1/MobileNetV2-Compression.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
os.chdir(REPO_DIR)

!pip install -q wandb tqdm

Tesla T4
(7, 5)
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 2.55 KiB | 871.00 KiB/s, done.
From https://github.com/lazylettuce1/MobileNetV2-Compression
   22a89dd..9ac338c  main       -> origin/main
Updating 22a89dd..9ac338c
Fast-forward
 compression_sweep.txt | 363 +++++++++++++++++++++-----------------------------
 quantize.py           |   7 +-
 wandb_sweep_cell.py   | 130 +++++++++++-------
 3 files changed, 234 insertions(+), 266 deletions(-)


In [56]:
from kaggle_secrets import UserSecretsClient; os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

DATA_DIR = "/kaggle/input/datasets/abhishekchavan2714/cifar10"
OUT_DIR  = "/kaggle/working/mobilenetv2-cifar/runs/baseline"

# Training Segment

In [ ]:
# input dataset, copy its checkpoint in before training starts
import os, shutil
PREV_CKPT = "/kaggle/working/MobileNetV2-Compression/outputs/baseline/best.pth"
os.makedirs(OUT_DIR, exist_ok=True)
resume_path = None
if os.path.exists(PREV_CKPT):
    resume_path = PREV_CKPT
    print("Found previous checkpoint, resuming from it")

import os
os.environ["WANDB_DIR"] = "/kaggle/working/wandb_logs"   # anywhere NOT inside your repo folder

from train import TrainConfig, run_training

cfg = TrainConfig(
    data_dir=DATA_DIR, 
    out_dir=OUT_DIR, 
    epochs=50,
    use_wandb=True, 
    resume=resume_path, 
    download=False,
    num_workers=0,
    # Add these overrides for fine-tuning:
    lr=0.001,          # Lower learning rate (default is 0.1)
    warmup_epochs=1    # Shorter warmup (default is 5)
)

history, model = run_training(cfg)

# Quantization Experiment Setup

In [58]:
from quantize import run_quantization_pipeline
baseline_ckpt = "/kaggle/working/MobileNetV2-Compression/outputs/baseline/best.pth"

quant_rpt = run_quantization_pipeline(
        ckpt_path=baseline_ckpt,
        data_dir=DATA_DIR,
        weight_bits=6,
        bias_bits=2,
        act_bits=8,
        calib_batches=40,
        sparsity=0.85,
        pruning_method='magnitude'
)

Manual Global Pruning: 1872176/2202560 parameters removed (85.00%). Threshold: 0.007583
fp32 accuracy: 93.52% | compressed accuracy: 92.85%
fp32 size: 8.66 MB | compressed size (theoretical): 0.71 MB
total compression: 12.15x
weight compression: 16.25x
activation compression: 4.00x


In [59]:
import itertools
from quantize import run_quantization_pipeline

weight_bits_list = [8, 6, 4]
act_bits_list = [8, 6, 4]
sparsity_list = [0.75, 0.85, 0.90]

all_reports = {}

for w_bits, a_bits, sparsity in itertools.product(weight_bits_list, act_bits_list, sparsity_list):
    print(f"\n==================================================================")
    print(f" Running: Weight Bits = {w_bits} | Act Bits = {a_bits} | Sparsity = {int(sparsity * 100)}%")
    print(f"==================================================================")
    
    quant_rpt = run_quantization_pipeline(
        ckpt_path=baseline_ckpt,
        data_dir=DATA_DIR,
        weight_bits=w_bits,
        bias_bits=2,
        act_bits=a_bits,
        calib_batches=40,
        sparsity=sparsity,  # <--- FIXED: changed 0.85 to the loop variable
        pruning_method='magnitude'
    )
    
    all_reports[(w_bits, a_bits, sparsity)] = quant_rpt


 Running: Weight Bits = 8 | Act Bits = 8 | Sparsity = 75%
Manual Global Pruning: 1651920/2202560 parameters removed (75.00%). Threshold: 0.003697
fp32 accuracy: 93.52% | compressed accuracy: 93.36%
fp32 size: 8.66 MB | compressed size (theoretical): 1.01 MB
total compression: 8.61x
weight compression: 10.37x
activation compression: 4.00x

 Running: Weight Bits = 8 | Act Bits = 8 | Sparsity = 85%
Manual Global Pruning: 1872176/2202560 parameters removed (85.00%). Threshold: 0.007583
fp32 accuracy: 93.52% | compressed accuracy: 92.87%
fp32 size: 8.66 MB | compressed size (theoretical): 0.80 MB
total compression: 10.88x
weight compression: 14.00x
activation compression: 4.00x

 Running: Weight Bits = 8 | Act Bits = 8 | Sparsity = 90%
Manual Global Pruning: 1982303/2202560 parameters removed (90.00%). Threshold: 0.012473
fp32 accuracy: 93.52% | compressed accuracy: 15.91%
fp32 size: 8.66 MB | compressed size (theoretical): 0.69 MB
total compression: 12.54x
weight compression: 16.97x
activ

# Quant Sweep for WANDB Parallel Coordinates Plot

In [44]:
"""
sweep.py — Run grid sweeps over weight bits, activation bits, and sparsities,
logging model size, accuracy, and compression metrics to Weights & Biases.
"""
import argparse
import itertools
import wandb
from quantize import run_quantization_pipeline


def run_wandb_sweep(
    ckpt_path,
    data_dir,
    weight_bits=(8, 6, 4, 2),
    act_bits=(8, 6, 4, 2),
    sparsities=(0.75, 0.85, 0.9),
    bias_bits=2,
    pruning_method="magnitude",
    calib_batches=40,
    project="mobilenetv2-cifar10-pruneplusquantv2",
):
    """
    Runs run_quantization_pipeline once per (weight_bits, act_bits, sparsity) combo,
    logging all outputs to W&B for Parallel Coordinates visualization.
    """
    for wbits, abits, sparsity in itertools.product(weight_bits, act_bits, sparsities):
        run = wandb.init(
            project=project,
            reinit="finish_previous",
            name=f"w{wbits}a{abits}_s{sparsity}_{pruning_method}",
            config={
                "weight_bits": wbits,
                "act_bits": abits,
                "bias_bits": bias_bits,
                "sparsity": sparsity,
                "calib_batches": calib_batches,
                "pruning_method": pruning_method,
            },
        )
        try:
            result = run_quantization_pipeline(
                ckpt_path=ckpt_path,
                data_dir=data_dir,
                weight_bits=wbits,
                act_bits=abits,
                bias_bits=bias_bits,
                calib_batches=calib_batches,
                sparsity=sparsity,
                pruning_method=pruning_method,
            )

            # Parse percentage string (e.g., "75.00%") if returned as string
            raw_sparsity = result.get("overall_weight_sparsity", sparsity)
            sparsity_pct = (
                float(raw_sparsity.rstrip("%"))
                if isinstance(raw_sparsity, str)
                else raw_sparsity * 100
            )

            wandb.log({
                "weight_bits": wbits,
                "act_bits": abits,
                "target_sparsity": sparsity,
                "overall_weight_sparsity_pct": sparsity_pct,
                "overall_compression_ratio": result["overall_compression_ratio"],
                "weights_compression_ratio": result["weights_compression_ratio"],
                "activation_compression_ratio": result["activation_compression_ratio"],
                "model_size_mb": result["quantized_sparse_total_mb"],
                "accuracy": result["accuracy"],
            })

        except Exception as e:
            print(f"FAILED at weight_bits={wbits}, act_bits={abits}, sparsity={sparsity}: {e}")
            wandb.log({"failed": 1})
        finally:
            run.finish()

    print("Sweep complete.")
    print("wandb -> your project -> Workspace -> Add panel -> Parallel Coordinates")
    print(
        "Suggested axes: weight_bits, act_bits, target_sparsity, "
        "overall_compression_ratio, weights_compression_ratio, accuracy"
    )


def _parse_args():
    parser = argparse.ArgumentParser(description="Run W&B sweep for pruning and quantization.")
    parser.add_argument("--ckpt-path", type=str, required=True, help="Path to baseline checkpoint (.pth)")
    parser.add_argument("--data-dir", type=str, default="./data")
    parser.add_argument("--weight-bits", type=int, nargs="+", default=[8, 6, 4, 2])
    parser.add_argument("--act-bits", type=int, nargs="+", default=[8, 6, 4, 2])
    parser.add_argument("--sparsities", type=float, nargs="+", default=[0.75, 0.85, 0.9])
    parser.add_argument("--bias-bits", type=int, default=16, help="Bit width for derived bias quantization")
    parser.add_argument("--pruning-method", type=str, default="magnitude", choices=["magnitude", "hessian"])
    parser.add_argument("--calib-batches", type=int, default=40)
    parser.add_argument("--project", type=str, default="mobilenetv2-cifar10-pruneplusquantv2")
    return parser.parse_args()


if __name__ == "__main__":
    args = _parse_args()
    run_wandb_sweep(
        ckpt_path=args.ckpt_path,
        data_dir=args.data_dir,
        weight_bits=args.weight_bits,
        act_bits=args.act_bits,
        sparsities=args.sparsities,
        bias_bits=args.bias_bits,
        pruning_method=args.pruning_method,
        calib_batches=args.calib_batches,
        project=args.project,
    )

In [45]:
run_wandb_sweep(
    ckpt_path="/kaggle/working/MobileNetV2-Compression/outputs/baseline/best.pth",
    data_dir=DATA_DIR,
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: lazylettuce1 (lazylettuce1-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Manual Global Pruning: 1651920/2202560 parameters removed (75.00%). Threshold: 0.003697
fp32 accuracy: 93.52% | compressed accuracy: 93.33%
fp32 size: 8.66 MB | compressed size (theoretical): 1.01 MB
total compression: 8.61x
weight compression: 10.37x
activation compression: 4.00x


accuracy,▁
act_bits,▁
activation_compression_ratio,▁
model_size_mb,▁
overall_compression_ratio,▁
target_sparsity,▁
weight_bits,▁
weights_compression_ratio,▁
accuracy,93.33
act_bits,8
activation_compression_ratio,4


Manual Global Pruning: 1872176/2202560 parameters removed (85.00%). Threshold: 0.007583
fp32 accuracy: 93.52% | compressed accuracy: 92.77%
fp32 size: 8.66 MB | compressed size (theoretical): 0.80 MB
total compression: 10.88x
weight compression: 14.00x
activation compression: 4.00x


accuracy,▁
act_bits,▁
activation_compression_ratio,▁
model_size_mb,▁
overall_compression_ratio,▁
target_sparsity,▁
weight_bits,▁
weights_compression_ratio,▁
accuracy,92.77
act_bits,8
activation_compression_ratio,4


...

Manual Global Pruning: 1872176/2202560 parameters removed (85.00%). Threshold: 0.007583
fp32 accuracy: 93.52% | compressed accuracy: 9.91%
fp32 size: 8.66 MB | compressed size (theoretical): 0.49 MB
total compression: 17.72x
weight compression: 28.70x
activation compression: 16.00x


accuracy,▁
act_bits,▁
activation_compression_ratio,▁
model_size_mb,▁
overall_compression_ratio,▁
target_sparsity,▁
weight_bits,▁
weights_compression_ratio,▁
accuracy,9.91
act_bits,2
activation_compression_ratio,16


Manual Global Pruning: 1982303/2202560 parameters removed (90.00%). Threshold: 0.012473
fp32 accuracy: 93.52% | compressed accuracy: 10.01%
fp32 size: 8.66 MB | compressed size (theoretical): 0.48 MB
total compression: 17.87x
weight compression: 29.10x
activation compression: 16.00x


accuracy,▁
act_bits,▁
activation_compression_ratio,▁
model_size_mb,▁
overall_compression_ratio,▁
target_sparsity,▁
weight_bits,▁
weights_compression_ratio,▁
accuracy,10.01
act_bits,2
activation_compression_ratio,16


Sweep complete.
wandb -> your project -> Workspace -> Add panel -> Parallel Coordinates
Suggested axes: weight_bits, act_bits, target_sparsity, overall_compression_ratio, weights_compression_ratio, accuracy
